# OCR Accuracy Measure Pipeline

**What this measures:** how faithfully each OCR engine reproduces the hand-transcribed ground truth (GT), independent of where the OCR engine happened to break its lines. 

**Important Terms**

CER = Character error rate

WER = Word error rate

**Summary**

Both sides are cut into the same normalized word stream (page headers removed, line-end hyphens joined, ditto glyphs unified). The GT stream is aligned to the OCR stream with a word-level edit-distance alignment that is *free at the OCR end*, so OCR text beyond what the GT covers is not counted as error. From that single alignment we report: **CER** (headline), **WER**, an error breakdown, recall by token type (numbers / names / dittos), per-line results, per-page results, and bootstrap confidence intervals for comparing engines.

In [2]:
import re, unicodedata
from collections import Counter
from dataclasses import dataclass

import numpy as np
import pandas as pd

pd.set_option('display.max_colwidth', 80)
pd.set_option('display.width', 200)

## 0. Paths

In [3]:
PATH_GT      = '../data/ground_truth/ground_truth_161_167.csv'   # column: Transcription
PATH_LINES   = '../data/ocr/lines.csv'                               # columns: tesseract_text, mocr_text, sources
PATH_IA_RAW  = '../data/ocr/A-O_raw_lines.csv'                       # column: raw_line (Internet Archive text, whole A-O volume)

## 1. Edit distance helpers
Uses `rapidfuzz` when installed (fast); falls back to a numpy implementation so the notebook still runs without it.

In [4]:
try:
    from rapidfuzz.distance import Levenshtein as _Lev
    def edit_distance(a: str, b: str) -> int:
        return _Lev.distance(a, b)
except ImportError:
    def edit_distance(a: str, b: str) -> int:
        if not a: return len(b)
        if not b: return len(a)
        A = np.frombuffer(a.encode('utf-32-le'), dtype=np.uint32)
        B = np.frombuffer(b.encode('utf-32-le'), dtype=np.uint32)
        j = np.arange(len(B) + 1)
        prev = j.copy()
        for i, ca in enumerate(A, 1):
            tmp = prev + 1
            tmp[1:] = np.minimum(tmp[1:], prev[:-1] + (B != ca))
            prev = np.minimum.accumulate(tmp - j) + j
        return int(prev[-1])

## 2. Normalization

In [5]:
HEADER_RE  = re.compile(r'BOSTON\s*[\[\(]?\s*\d{3,4}\s*[\]\)]?\s*D\w{4,9}', re.I)   # tolerant of OCR typos (DIEECTORT)
PAGENUM_RE = re.compile(r'^\W*\d{3,4}\W*$')
LEAD_DITTO = re.compile(r'^(?:[«»"\'`‘’“”,.\-~*•:;|]+\s*)+')
FOLD = str.maketrans({'‘': "'", '’': "'", '“': '"', '”': '"', '–': '—', '−': '-'})

def is_page_artifact(line: str) -> bool:
    return bool(HEADER_RE.search(line)) or bool(PAGENUM_RE.match(line))

def canon_line(raw, profile='standard'):
    """Return a cleaned line string, or None if the line is empty / a page artifact."""
    s = ' '.join(str(raw if raw is not None and raw == raw else '').split())
    if not s or is_page_artifact(s):
        return None
    if profile == 'strict':
        return s
    s = unicodedata.normalize('NFC', s)
    m = LEAD_DITTO.match(s)
    if m:
        rest = s[m.end():].strip()
        s = '" ' + rest if rest else '"'
    s = s.translate(FOLD)
    s = re.sub(r'\s*—\s*', ' — ', s)
    return ' '.join(s.split())

def finalize_token(tok, profile):
    if profile == 'no_ditto':
        return '' if tok == '"' else tok
    if profile != 'lenient' or tok == '"':
        return tok
    return re.sub(r'[^\w&]', '', tok).casefold()

@dataclass
class Stream:
    tokens: list      # normalized word tokens
    line: list        # source line index (position in the input list) for each token
    page: list        # page number for each token (or None)
    n_artifacts: int  # header / page-number / empty lines dropped

def prepare(lines, profile='standard', pages=None, join_hyphen=True) -> Stream:
    toks, tline, tpage = [], [], []
    dropped, pending = 0, None

    def emit(t, ln, pg):
        t = finalize_token(t, profile)
        if t:
            toks.append(t); tline.append(ln); tpage.append(pg)

    for i, raw in enumerate(lines):
        pg = pages[i] if pages is not None else None
        s = canon_line(raw, profile)
        if s is None:
            dropped += 1
            continue
        words = s.split()
        if pending:
            emit(pending[0] + words.pop(0), pending[1], pending[2])
            pending = None
        if join_hyphen and words and len(words[-1]) > 1 and words[-1].endswith('-') and words[-1][-2].isalnum():
            pending = (words.pop()[:-1], i, pg)
        for w in words:
            emit(w, i, pg)
    if pending:
        emit(pending[0] + '-', pending[1], pending[2])
    return Stream(toks, tline, tpage, dropped)

## 3. Alignment (word level, free OCR end)

In [6]:
def align(gt_tok, ocr_tok, window=1.5, slack=50):
    """Align GT tokens to a prefix of the OCR tokens.
    Returns (ops, end) where ops is a list of (tag, gt_idx, ocr_idx) with tag in
    {'equal','sub','del','ins'} ('del' = GT word missing from OCR, 'ins' = extra OCR word),
    and end = number of OCR tokens consumed."""
    vocab = {}
    g = np.array([vocab.setdefault(t, len(vocab)) for t in gt_tok])
    m = min(len(ocr_tok), int(window * len(g)) + slack)
    o = np.array([vocab.setdefault(t, len(vocab)) for t in ocr_tok[:m]])
    n = len(g)
    j = np.arange(m + 1)
    prev = j.copy()
    ptr = np.zeros((n + 1, m + 1), dtype=np.int8)   # 0 diag, 1 up (del), 2 left (ins)
    ptr[0, 1:] = 2
    for i in range(1, n + 1):
        diag = prev[:-1] + (o != g[i - 1])
        up = prev + 1
        tmp = up.copy()
        tmp[1:] = np.minimum(tmp[1:], diag)
        cur = np.minimum.accumulate(tmp - j) + j
        p = np.full(m + 1, 2, dtype=np.int8)
        p[up == cur] = 1
        p[1:][diag == cur[1:]] = 0
        ptr[i] = p
        prev = cur
    end = int(np.argmin(prev))
    ops, i, jj = [], n, end
    while i > 0 or jj > 0:
        p = ptr[i, jj] if i > 0 else 2
        if p == 0:
            ops.append(('equal' if g[i - 1] == o[jj - 1] else 'sub', i - 1, jj - 1)); i -= 1; jj -= 1
        elif p == 1:
            ops.append(('del', i - 1, None)); i -= 1
        else:
            ops.append(('ins', None, jj - 1)); jj -= 1
    ops.reverse()
    return ops, end, m

## 4. Evaluate one OCR source

In [7]:
@dataclass
class Result:
    name: str
    profile: str
    gt: Stream
    ocr: Stream
    ops: list
    end: int
    counts: dict
    cer: float
    wer: float
    lines: pd.DataFrame

def tok_class(t):
    if t == '"': return 'ditto "'
    if any(c.isdigit() for c in t): return 'number'
    if t[:1].isupper(): return 'Capitalised (names, districts)'
    if t[:1].isalpha(): return 'lowercase (occupations, abbrevs)'
    return 'symbol'

def evaluate(gt_lines, ocr_lines, name='ocr', profile='standard', pages=None, join_hyphen=True) -> Result:
    gt  = prepare(gt_lines,  profile, None,  join_hyphen)
    ocr = prepare(ocr_lines, profile, pages, join_hyphen)
    ops, end, m = align(gt.tokens, ocr.tokens)
    if len(ocr.tokens) > m and end >= m - 5:
        print(
        f'[{name}] WARNING: alignment ended near the edge of the OCR window;'
        ' GT may extend further than assumed.'
        )

    c = Counter(tag for tag, _, _ in ops)
    N = len(gt.tokens)
    counts = dict(N=N, equal=c['equal'], sub=c['sub'], dele=c['del'], ins=c['ins'])
    wer = (c['sub'] + c['del'] + c['ins']) / N

    gt_text  = ' '.join(gt.tokens)
    ocr_text = ' '.join(ocr.tokens[:end])
    cer = edit_distance(gt_text, ocr_text) / len(gt_text)

    # ---- per-GT-line view (errors attributed to the GT line they fall in)
    per = {}
    cur, pend = None, []
    def bucket(l):
        return per.setdefault(l, dict(gt=[], ocr=[], err=0, page=None))
    for tag, gi, oi in ops:
        if gi is not None:
            cur = gt.line[gi]
            b = bucket(cur)
            b['gt'].append(gt.tokens[gi])
            for t in pend:                       # leading insertions go to the first GT line
                b['ocr'].append(ocr.tokens[t]); b['err'] += 1
            pend = []
        if tag == 'equal' or tag == 'sub':
            b = bucket(cur); b['ocr'].append(ocr.tokens[oi])
            b['err'] += tag == 'sub'
            if b['page'] is None: b['page'] = ocr.page[oi]
        elif tag == 'del':
            bucket(cur)['err'] += 1
        else:
            if cur is None: pend.append(oi)
            else:
                b = bucket(cur); b['ocr'].append(ocr.tokens[oi]); b['err'] += 1
    rows = []
    for l, b in sorted(per.items()):
        g_txt, o_txt = ' '.join(b['gt']), ' '.join(b['ocr'])
        rows.append(dict(gt_line=l, page=b['page'], gt_text=g_txt, ocr_text=o_txt,
                         word_errors=b['err'], gt_chars=len(g_txt), char_dist=edit_distance(g_txt, o_txt)))
    lines = pd.DataFrame(rows)
    if lines['page'].notna().any():
        lines['page'] = lines['page'].ffill().bfill().astype(int)
    return Result(name, profile, gt, ocr, ops, end, counts, cer, wer, lines)

## 5. Reporting helpers

In [8]:
def summary_row(r: Result):
    N = r.counts['N']
    return {
        'source': r.name, 'profile': r.profile,
        'CER %': 100 * r.cer, 'WER %': 100 * r.wer,
        'char acc %': 100 * (1 - r.cer), 'word acc %': 100 * (1 - r.wer),
        'sub %': 100 * r.counts['sub'] / N, 'del %': 100 * r.counts['dele'] / N, 'ins %': 100 * r.counts['ins'] / N,
        'perfect lines %': 100 * (r.lines.word_errors == 0).mean(),
        'GT words': N, 'OCR words used': r.end, 'OCR words total': len(r.ocr.tokens),
    }

def class_recall(r: Result) -> pd.DataFrame:
    """Share of GT words of each type that the OCR reproduced exactly."""
    hit = Counter(); tot = Counter()
    for tag, gi, _ in r.ops:
        if gi is None: continue
        k = tok_class(r.gt.tokens[gi]); tot[k] += 1; hit[k] += tag == 'equal'
    return pd.DataFrame({'GT words': tot, 'exact match %': {k: 100 * hit[k] / tot[k] for k in tot}}).sort_values('GT words', ascending=False)

def per_page(r: Result) -> pd.DataFrame:
    d = r.lines.dropna(subset=['page'])
    if d.empty: return pd.DataFrame()
    g = d.groupby('page').agg(lines=('gt_line', 'size'), gt_chars=('gt_chars', 'sum'),
                              char_dist=('char_dist', 'sum'), perfect=('word_errors', lambda s: (s == 0).mean() * 100))
    g['line-conditioned CER %'] = 100 * g.char_dist / g.gt_chars
    return g[['lines', 'line-conditioned CER %', 'perfect']].rename(columns={'perfect': 'perfect lines %'})

def per_block(r: Result, size=50) -> pd.Series:
    """Line-conditioned CER for consecutive blocks of GT lines. Reveals localized failures (dropped or re-ordered text)."""
    d = r.lines.assign(block=lambda x: x.gt_line // size * size)
    g = d.groupby('block').agg(dist=('char_dist', 'sum'), n=('gt_chars', 'sum'))
    return 100 * g.dist / g.n

def top_confusions(r: Result, k=15):
    subs, dels, ins = Counter(), Counter(), Counter()
    for tag, gi, oi in r.ops:
        if tag == 'sub': subs[(r.gt.tokens[gi], r.ocr.tokens[oi])] += 1
        elif tag == 'del': dels[r.gt.tokens[gi]] += 1
        elif tag == 'ins': ins[r.ocr.tokens[oi]] += 1
    return (pd.DataFrame(subs.most_common(k), columns=['(GT, OCR)', 'count']),
            pd.DataFrame(dels.most_common(k), columns=['missed GT word', 'count']),
            pd.DataFrame(ins.most_common(k),  columns=['spurious OCR word', 'count']))

def worst_lines(r: Result, k=15):
    d = r.lines
    return d.sort_values('char_dist', ascending=False).head(k)[['gt_line', 'page', 'gt_text', 'ocr_text', 'word_errors', 'char_dist']]

def bootstrap_cer(results: dict, n_boot=2000, seed=0):
    """Resample GT lines with replacement. Paired across engines so differences are meaningful.
    CER here is line-conditioned (sum of per-line distances / GT chars), so it runs a little higher than the headline CER (line attribution of displaced text); use it for *comparisons*, not as the headline."""
    rng = np.random.default_rng(seed)
    names = list(results)
    base = results[names[0]].lines.gt_line.values
    D = np.column_stack([results[n].lines.set_index('gt_line').reindex(base).char_dist.values for n in names])
    L = results[names[0]].lines.gt_chars.values
    idx = rng.integers(0, len(base), size=(n_boot, len(base)))
    boot = D[idx].sum(1) / L[idx].sum(1)[:, None]           # (n_boot, n_engines)
    out = pd.DataFrame({'CER % (line-cond.)': 100 * D.sum(0) / L.sum(),
                        '95% CI low': 100 * np.percentile(boot, 2.5, 0),
                        '95% CI high': 100 * np.percentile(boot, 97.5, 0)}, index=names)
    diffs = {}
    for a in range(len(names)):
        for b in range(a + 1, len(names)):
            d = 100 * (boot[:, a] - boot[:, b])
            diffs[f'{names[a]} - {names[b]}'] = dict(mean_diff=d.mean(), ci_low=np.percentile(d, 2.5), ci_high=np.percentile(d, 97.5),
                                                      significant=not (np.percentile(d, 2.5) <= 0 <= np.percentile(d, 97.5)))
    return out, pd.DataFrame(diffs).T

## 6. Load data

In [11]:
gt_df = pd.read_csv(PATH_GT)
gt_lines = gt_df['Transcription'].fillna('').astype(str).tolist()

ocr_df = pd.read_csv(PATH_LINES)
pages = ocr_df['sources'].str.extract(r'/page/(\d+)')[0].astype(int).tolist()

ia_df = pd.read_csv(PATH_IA_RAW, nrows=4 * len(gt_lines))      # only the start of the volume is needed
SOURCES = {
    'Tesseract':        dict(lines=ocr_df['tesseract_text'].fillna('').astype(str).tolist(), pages=pages),
    'MOCR':             dict(lines=ocr_df['mocr_text'].fillna('').astype(str).tolist(),      pages=pages),
    'Internet Archive': dict(lines=ia_df['raw_line'].fillna('').astype(str).tolist(),        pages=None),
    'easyocr':          dict(lines=ocr_df['easyocr_text'].fillna('').astype(str).tolist(), pages=pages),
    'paddleocr':        dict(lines=ocr_df['paddleocr_text'].fillna('').astype(str).tolist(), pages=pages),
    'qwen':             dict(lines=ocr_df['qwen_text'].fillna('').astype(str).tolist(), pages=pages)

}
print(f'GT lines: {len(gt_lines)}   OCR lines: {len(ocr_df)} (pages {min(pages)}-{max(pages)})   IA lines loaded: {len(ia_df)}')

GT lines: 1369   OCR lines: 1573 (pages 161-167)   IA lines loaded: 5476


## 7. Headline results (`standard` profile)

In [12]:
results = {name: evaluate(gt_lines, s['lines'], name, 'standard', s['pages']) for name, s in SOURCES.items()}
headline = pd.DataFrame([summary_row(r) for r in results.values()]).set_index('source').drop(columns='profile')
headline.round(2)

,CER %,WER %,char acc %,word acc %,sub %,del %,ins %,perfect lines %,GT words,OCR words used,OCR words total
source,,,,,,,,,,,
Tesseract,7.01,14.20,92.99,85.80,3.57,6.61,4.01,35.22,10417,10146,10146
MOCR,9.55,15.69,90.45,84.31,2.23,6.50,6.96,38.74,10417,10465,10465
Internet Archive,31.44,39.27,68.56,60.73,9.22,23.11,6.95,32.13,10417,8734,38057
easyocr,8.56,20.40,91.44,79.60,9.58,6.96,3.86,24.21,10417,10094,10095
paddleocr,7.50,15.09,92.50,84.91,4.30,7.26,3.53,34.34,10417,10029,10029
qwen,33.77,40.01,66.23,59.99,4.55,6.78,28.68,30.01,10417,12699,12857


**How to read it:** CER/WER = (substitutions + deletions + insertions) / GT size. `perfect lines %` is the share of GT lines whose every word was reproduced exactly.

### Where does the GT end inside each OCR source?

In [13]:
for name, r in results.items():
    last = r.ocr.line[r.end - 1]
    pg = r.ocr.page[r.end - 1]
    print(f'{name:17s} GT consumed OCR words 0-{r.end} (input line {last}' + (f', page {pg})' if pg else ')'))

Tesseract         GT consumed OCR words 0-10146 (input line 1572, page 167)
MOCR              GT consumed OCR words 0-10465 (input line 1572, page 167)
Internet Archive  GT consumed OCR words 0-8734 (input line 1274)
easyocr           GT consumed OCR words 0-10094 (input line 1572, page 167)
paddleocr         GT consumed OCR words 0-10029 (input line 1572, page 167)
qwen              GT consumed OCR words 0-12699 (input line 1562, page 167)


### Sensitivity to normalization

In [14]:
PROFILES = ['strict', 'standard', 'no_ditto', 'lenient']

In [15]:
sens = pd.DataFrame([summary_row(evaluate(gt_lines, s['lines'], n, p, s['pages']))
                     for n, s in SOURCES.items() for p in PROFILES])
sens.pivot(index='source', columns='profile', values='CER %').round(2)[PROFILES]

profile,strict,standard,no_ditto,lenient
source,,,,
Internet Archive,31.98,31.44,30.61,30.74
MOCR,22.11,9.55,6.86,9.36
Tesseract,7.64,7.01,4.46,6.59
easyocr,8.73,8.56,6.69,8.13
paddleocr,7.65,7.50,4.70,7.09
qwen,44.08,33.77,32.42,31.84


In [16]:
sens.pivot(index='source', columns='profile', values='WER %').round(2)[PROFILES]

profile,strict,standard,no_ditto,lenient
source,,,,
Internet Archive,41.58,39.27,37.45,37.92
MOCR,40.81,15.69,9.31,15.22
Tesseract,17.13,14.20,8.36,13.18
easyocr,22.01,20.40,15.91,19.15
paddleocr,16.15,15.09,8.68,14.24
qwen,44.21,40.01,36.45,39.09


### What kind of words are being missed?

In [17]:
pd.concat({n: class_recall(r)['exact match %'] for n, r in results.items()}, axis=1).round(1).join(
    class_recall(results['Tesseract'])['GT words'])

,Tesseract,MOCR,Internet Archive,easyocr,paddleocr,qwen,GT words
"Capitalised (names, districts)",94.7,96.8,69.4,87.3,94.4,94.6,4806
"lowercase (occupations, abbrevs)",97.4,98.0,71.4,92.4,95.7,95.3,2937
number,96.6,98.0,68.3,93.3,94.3,96.3,1480
"ditto """,30.7,30.5,48.0,25.1,28.8,26.5,932
symbol,87.4,92.0,61.5,64.1,77.9,84.7,262


### Per page (Tesseract / MOCR only; the IA text has no page markers)

In [18]:
pd.concat({n: per_page(results[n])['line-conditioned CER %'] for n in ('Tesseract', 'MOCR')}, axis=1).round(2)

,Tesseract,MOCR
page,,
161,5.18,8.18
162,9.82,10.10
163,7.11,10.86
164,10.20,10.13
165,7.29,14.44
166,5.15,8.59
167,10.74,9.02


### Localized failures: CER per block of 50 GT lines

In [19]:
pd.concat({n: per_block(r) for n, r in results.items()}, axis=1).round(1)

,Tesseract,MOCR,Internet Archive,easyocr,paddleocr,qwen
block,,,,,,
0,5.6,12.3,12.6,11.2,7.4,50.1
50,6.5,3.9,29.1,6.5,5.4,8.2
100,10.2,13.1,7.6,13.6,13.4,67.9
150,15.2,13.8,94.5,14.4,14.9,120.7
200,5.9,5.8,27.2,10.2,9.1,45.1
250,7.9,6.8,71.5,8.6,6.9,10.2
300,13.0,19.7,20.0,18.9,13.6,48.9
350,5.1,8.8,12.5,10.4,8.5,11.1
400,6.0,4.1,25.9,5.5,2.6,23.5


### Are the differences between engines real?

In [20]:
ci, diffs = bootstrap_cer(results)
display(ci.round(2)); display(diffs.round(2))

,CER % (line-cond.),95% CI low,95% CI high
Tesseract,8.17,7.13,9.34
MOCR,10.38,8.77,12.23
Internet Archive,33.74,30.11,37.93
easyocr,10.62,9.68,11.62
paddleocr,8.99,8.13,9.92
qwen,35.61,29.08,43.27


,mean_diff,ci_low,ci_high,significant
Tesseract - MOCR,-2.201938,-4.083466,-0.591891,True
Tesseract - Internet Archive,-25.57937,-29.856211,-21.810761,True
Tesseract - easyocr,-2.455512,-3.329122,-1.542389,True
Tesseract - paddleocr,-0.829214,-1.631526,0.010657,False
Tesseract - qwen,-27.506512,-35.067108,-21.081045,True
MOCR - Internet Archive,-23.377433,-27.89554,-19.291517,True
MOCR - easyocr,-0.253574,-1.754648,1.646959,False
MOCR - paddleocr,1.372724,-0.11502,3.245756,False
MOCR - qwen,-25.304575,-32.791954,-19.03009,True
Internet Archive - easyocr,23.123858,19.279997,27.419436,True


## 8. Error analysis

In [21]:
R = results['Tesseract']          # change to 'MOCR' or 'Internet Archive'
worst_lines(R, 15)

,gt_line,page,gt_text,ocr_text,word_errors,char_dist
1287,1293,167,""" Henry A foreman h 17 Taft Dor",""" Henry A foreman h 17 Taft Dor hihi 3 = ES = Re RREk RioiRURIR OR = RORORSR...",29,113
1174,1179,167,ADAMS EPHRAIM & CO Inc T C Adams treas,ADAMS EPHRAIM & CO Ine T C Adams treas RRCRCR RUR RR. RRS I E = FF RVR BR 2 ...,27,86
411,413,163,""" Thomas longshoreman bds 50 Kendall",""" Thomas longshoreman bds 50 Kendall | R.B RRR. RR ARR eE & BEE EER E 13 ROR...",20,71
845,848,165,1301 Commonwealth av Alls,"1301 Commonwealth av Alls "" = 2 Rae = "" = s = "" iS SRR ATR = *h "" "" 22 RR RR...",23,63
1052,1057,166,""" Blanche dressmaker 29 Hamilton Dor bds do","3 Blanche dressmaker 29 Hamilton Dor bds do "" "" "" = = = "" = = "" = RRR OB "" =...",22,61
27,28,161,""" Solomon E vice pres 60 State rm 316 bds 93",""" Solomon ne ie pres 60 State rm 316 bds 93 Henl Sophie or h 578 Newbury S E...",14,58
194,195,162,""" E D Company printers 181 Mass av","E D Company printers 181 Mass av Ab "" "" AB £2 = 3 RIMRTR ES 2 = = ROIRAB RER...",15,51
566,569,164,""" Joel sub carrier h 43 Hollander Rox","Joel sub carrier P O, sta A bds 12 Marble Rox Louis cigarmaker h 43 Hollande...",11,48
467,470,163,""" Alfred J clerk 131 State rm 216 h 231 High""",""" Alfred J clerk 131 State rm 216 h 231 High"" William H collector 501 Wash h...",9,44
887,890,165,Ackhurst Hardy W pianotuner rms 88 Pembroke,Ackhurst Hardy W pianotuner rms 88 Pembroke Ackiss C L clerk 117 Franklin h ...,9,42


In [22]:
subs, dels, ins = top_confusions(R)
display(subs); display(dels); display(ins)

,"(GT, OCR)",count
0,"(L, LL)",7
1,"("", :)",6
2,"(S, 8S)",4
3,"(see, See)",3
4,"(L, Lh)",3
5,"("", or)",3
6,"(av, ay)",3
7,"(bds, hds)",3
8,"("", i)",2
9,"("", ;)",2


,missed GT word,count
0,"""",574
1,B,5
2,av,4
3,Dor,3
4,h,3
5,B'line,3
6,R,3
7,A,2
8,sq,2
9,Alls,2


,spurious OCR word,count
0,"""",65
1,=,48
2,h,11
3,RR,10
4,2,7
5,3,7
6,Ab,7
7,BR,6
8,ES,6
9,E,5
